In [ ]:
from langchain_core.tools  import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage,ToolMessage
import requests, json


In [ ]:
@tool
def getcurrencyrates(base_currency: str, desired_currency:str)->float:
    """Currency rates provider, provides the latest currency exchange rates"""

    url=f"https://v6.exchangerate-api.com/v6/your_api_key/pair/{base_currency.upper()}/{desired_currency.upper()}"

    response=requests.get(url)
    return response.json()



In [ ]:

@tool
def convert(base_currency : int, conversion_rate:float)->float:
    """This tool is mainly used to convert the data from the given base currency to the desired currency"""

    return base_currency*conversion_rate

In [ ]:
tool_map={
    "getcurrencyrates":getcurrencyrates,
    "convert":convert
}

tool_map["getcurrencyrates"]

In [ ]:
llm=ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")
model_with_tool=llm.bind_tools(list(tool_map.values()))

In [ ]:
def calculatecurrencyapp(query:str):
    message=[HumanMessage(content=query)]

    max_iterations=5

    for _ in range(max_iterations):
        response=model_with_tool.invoke(message)
        message.append(response)

        if not response.tool_calls:
            print(response.text)
            return response
        
        for tool_call in response.tool_calls:
            tool_name=tool_call['name']
            tool_id=tool_call['id']
            tool_args=tool_call['args']

            selected_tool=tool_map[tool_name]
            tool_output=selected_tool.invoke(tool_args)

            tool_message=ToolMessage(
                content=str(tool_output),
                name=tool_name,
                tool_call_id=tool_id 
            )
            message.append(tool_message)
            


In [ ]:
calculatecurrencyapp(query="What is the current rate of inr in us dollars and uk")